# AMEX Enterprise Credit Risk Platform
## Notebook 46 -- Roll-Rate Modeling: Business Understanding & Policy
### Phase 3 . Problem Statement 8: Roll-Rate Modeling

CRISP-DM stage: **Business Understanding**. Depends on Problem 1 Notebooks 01/02/05/08, Problem 4's real
severity-scoring bundle (Notebook 28), and Problem 6's real persisted trailing-window model (Notebook 40).

**What this notebook does (real, computed on your machine when you run it):**
- Reuses Problem 4's real, vetted delinquency-severity feature universe (recovered from its real
  `severity_scoring_bundle.json`) as the monitored base-column set -- not a fresh, unvetted selection
- Explains, with a documented methodological note, why Problem 4's own severity tiers (fit ONCE per
  customer from whole-history summary features) cannot be reused directly for a transition model, and
  why Problem 8 must define a fresh, analogous PER-STATEMENT severity score instead
- States the 3-state policy (Low Severity / Moderate Severity / Severe, reusing Problem 4's tier names for
  cross-platform continuity) and the tertile cut-percentile convention (the actual cut VALUES are a real,
  measured output of Notebook 47, not assumed here)
- Sets KPI targets appropriate to a Markov transition-probability model: Problem 4's own real monotonicity
  KPI (reused verbatim, applied to each customer's latest observed state) as the primary hard gate, plus a
  new transition-matrix coherence check (persistence in the worst state must exceed a single-step jump
  from the best state) with no prior-problem precedent to reuse
- Records an honest, exploratory (not production-validating) plan to stratify the final transition by
  Problem 6's real persisted dynamic-PD score, explicitly carrying forward Problem 6's own honest
  NOT-RECOMMENDED status rather than treating its score as validated
- Measures the real per-customer statement-count distribution and real transition-pair eligibility coverage
  from the live raw CSV, writes `roll_rate_policy.json`

**What this notebook does NOT do:** it does not compute any severity score, cutpoint, or transition
probability itself -- those are real, measured outputs of Notebook 47 (Modeling), never assumed here.

Zero-fabrication: every threshold in this notebook is either reused verbatim from an earlier notebook's
real, already-validated output (Problem 4's tier names and monotonicity KPI, Problem 1's real champion AUC,
Problem 6's real winning window and recommendation status) or explicitly labeled ASSUMPTION and editable.
Nothing is guessed.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05),
#            PROBLEM 4'S REAL SEVERITY BUNDLE (NOTEBOOK 28), AND PROBLEM 6'S
#            REAL TRAILING-WINDOW MODEL (NOTEBOOK 40)
# =============================================================================
import os
import sys
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05, 28, 38-40")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB08_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_08_summary.json"
NB28_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_28_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"
NB40_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_40_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB08_SUMMARY_PATH, "run 08_basel_ifrs9_mapping.ipynb first (real EAD/LGD assumptions inherited, "
                         "not re-guessed)"),
    (NB28_SUMMARY_PATH, "run 28_validation_deployment.ipynb (Problem 4) first -- Problem 8 depends on "
                         "Problem 4 per the master plan, reusing its real, vetted severity-feature "
                         "universe rather than re-deriving one"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first"),
    (NB40_SUMMARY_PATH, "run 40_dynamic_behavioral_scoring_validation_deployment.ipynb first -- Problem 8 "
                         "depends on Problem 6 per the master plan, reusing its real persisted trailing-"
                         "window model as an exploratory covariate for stratifying the transition matrix"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(NB28_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB28_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)
with open(NB40_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB40_SUMMARY = json.load(f)

# --- Problem 4's real severity-scoring bundle: the source of truth for which
#     raw D_* columns this platform has already vetted as delinquency-severity
#     indicators (fit via abs(correlation-with-target)-weighted z-scores over
#     Problem 1's real engineered feature store). Problem 8 reuses this same
#     FEATURE UNIVERSE (which base columns to look at), not Problem 4's fitted
#     weights/cutpoints themselves -- those were fit on a CUSTOMER-level
#     population (one row per customer, whole-history summary features);
#     Problem 8 operates at STATEMENT level (one row per customer per
#     statement) and must fit its own weights/cutpoints fresh on that
#     population in Notebook 47, or the fit would not match the population
#     it's applied to. This distinction is recorded explicitly below and in
#     the policy artifact so it's never silently conflated later. ---
if "severity_scoring_bundle.json" not in NB28_SUMMARY.get("output_files", {}):
    raise KeyError("notebook_28_summary.json has no 'severity_scoring_bundle.json' entry under output_files.")
SEVERITY_BUNDLE_PATH = Path(NB28_SUMMARY["output_files"]["severity_scoring_bundle.json"])
if not SEVERITY_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"{SEVERITY_BUNDLE_PATH} not found.\nFix: re-run Notebook 28 (Problem 4).")
with open(SEVERITY_BUNDLE_PATH, "r", encoding="utf-8") as f:
    P4_SEVERITY_BUNDLE = json.load(f)
P4_FEATURE_LIST = sorted(P4_SEVERITY_BUNDLE["features"])
P4_TIER_ORDER = P4_SEVERITY_BUNDLE["tier_order"]

# --- Problem 6's real persisted trailing-window model -- reused ONLY as an
#     exploratory covariate for stratifying the transition matrix (Section 8),
#     never as a validated production signal. Problem 6's own honest
#     recommendation status is carried through unchanged. ---
P6_WINNING_W = NB40_SUMMARY["winning_w"]
P6_RECOMMENDED_FOR_PRODUCTION = NB40_SUMMARY["recommended_for_production"]
P6_MODEL_PATH = Path(NB40_SUMMARY["model_path"])
P6_PREPROCESSING_PATH = Path(NB40_SUMMARY["preprocessing_path"])
for _p, _label in [(P6_MODEL_PATH, "Problem 6's persisted model"),
                    (P6_PREPROCESSING_PATH, "Problem 6's preprocessing artifacts")]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found ({_label}).\nFix: re-run Notebook 40 (Problem 6).")

EAD_PER_ACCOUNT_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
LGD_ASSUMPTION = NB08_SUMMARY["lgd_assumption"]

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]
_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")

if "roll_rate_policy" in PILLAR_DIRS:
    RR_POLICY_DIR = PILLAR_DIRS["roll_rate_policy"]
else:
    RR_POLICY_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem8_Roll_Rate_Modeling" / "policy"
    )
    print(f"NOTE: 'roll_rate_policy' not in pillar_dirs -- using fallback: {RR_POLICY_DIR}")
RR_POLICY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Loaded config from                       : {CONFIG_PATH}")
print(f"Reused Problem 4's real severity bundle  : {SEVERITY_BUNDLE_PATH}")
print(f"Problem 4 tier order (reused for continuity): {P4_TIER_ORDER}")
print(f"Problem 6 winning window / recommended    : W={P6_WINNING_W} / {P6_RECOMMENDED_FOR_PRODUCTION}")
print(f"Champion architecture (Problem 1, measured): {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference) : {FULL_HISTORY_AUC}")
print(f"EAD/LGD (Notebook 08, inherited)           : ${EAD_PER_ACCOUNT_USD:,} / {LGD_ASSUMPTION:.0%}")
print(f"Reused feature universe (Problem 4's real severity bundle): {len(P4_FEATURE_LIST)} features")
print(f"Policy artifacts will be written under: {RR_POLICY_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import psutil
except ImportError:
    missing.append("psutil")
if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}"
    )


def _rss_gb() -> float:
    return psutil.Process().memory_info().rss / 1e9


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break
if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n" + "\n".join(f"  - {c}" for c in _raw_candidates)
    )
RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(f"{RAW_TRAIN_LABELS_PATH} not found.")

print(f"Raw train_data.csv  : {RAW_TRAIN_DATA_PATH}")
print(f"Raw train_labels.csv: {RAW_TRAIN_LABELS_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA DETECTION -- BASE RAW COLUMNS FOR PER-STATEMENT
#            SEVERITY SCORING (REUSES PROBLEM 4'S REAL VETTED FEATURE UNIVERSE)
# =============================================================================
_section("SECTION 4: Live Schema Detection -- Base Raw Columns for Per-Statement Severity Scoring")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

# --- Problem 8 needs each statement's OWN raw column values (a snapshot per
#     row), not the whole-history summary features (_last/_trend_delta/
#     _trend_slope) Problem 4 fit its tiers on. Recovers the underlying base
#     D_* columns from Problem 4's real feature list via the same suffix-
#     stripping method Notebooks 39/42 established, so the monitored universe
#     is provably the same columns already vetted for delinquency-severity
#     signal in Problem 4 -- not a fresh, unvetted selection. ---
_SUFFIXES = ("_trend_slope", "_trend_delta", "_last")
CANDIDATE_FEATURES = set()
for _feat in P4_FEATURE_LIST:
    for _suf in _SUFFIXES:
        if _feat.endswith(_suf):
            CANDIDATE_FEATURES.add(_feat[: -len(_suf)])
            break

_missing_cols = CANDIDATE_FEATURES - _header_cols
if _missing_cols:
    raise RuntimeError(
        f"{len(_missing_cols)} candidate column(s) are not present in the real raw CSV header: "
        f"{sorted(_missing_cols)}\nFix: investigate before proceeding rather than silently dropping columns."
    )
CANDIDATE_FEATURES = sorted(CANDIDATE_FEATURES)

# --- Cross-check against Problem 6's real feature universe, when its own
#     policy is available -- both problems ultimately trace back to the SAME
#     Problem 4 vetted list, so a mismatch here would indicate a real drift
#     between the two problems' feature selection, not an expected outcome. ---
try:
    P6_POLICY_PATH = Path(NB38_SUMMARY["policy_path"])
    with open(P6_POLICY_PATH, "r", encoding="utf-8") as f:
        _p6_policy = json.load(f)
    _p6_base_features = set()
    for _feat in _p6_policy["feature_space"]["features"]:
        for _suf in _SUFFIXES:
            if _feat.endswith(_suf):
                _p6_base_features.add(_feat[: -len(_suf)])
                break
    _feature_universe_matches_p6 = set(CANDIDATE_FEATURES) == _p6_base_features
except (FileNotFoundError, KeyError):
    _feature_universe_matches_p6 = None

print(f"Real base D_* columns monitored for per-statement severity scoring: {len(CANDIDATE_FEATURES)}")
print(f"Sample: {CANDIDATE_FEATURES[:5]}")
print(f"Feature universe matches Problem 6's real base list: {_feature_universe_matches_p6}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: REAL PER-CUSTOMER STATEMENT-COUNT DISTRIBUTION (FOR TRANSITION-
#            PAIR FEASIBILITY)
# =============================================================================
_section("SECTION 5: Real Per-Customer Statement-Count Distribution")

print("Reading real per-statement (raw, pre-aggregation) data from: " + str(RAW_TRAIN_DATA_PATH))
_t0 = time.time()
_statement_counts = (
    pl.scan_csv(RAW_TRAIN_DATA_PATH)
    .select(pl.col("customer_ID"))
    .group_by("customer_ID")
    .agg(pl.len().alias("n_statements"))
    .collect()
)
print(f"Grouped in {time.time() - _t0:.1f}s. Process RSS: {_rss_gb():.2f} GB")
_n_customers = _statement_counts.height
_counts_series = _statement_counts["n_statements"]
STATEMENT_COUNT_STATS = {
    "n_customers": _n_customers,
    "min": int(_counts_series.min()),
    "p10": float(_counts_series.quantile(0.10)),
    "p25": float(_counts_series.quantile(0.25)),
    "median": float(_counts_series.median()),
    "mean": float(_counts_series.mean()),
    "max": int(_counts_series.max()),
}
for _k in ("min", "p10", "p25", "median", "mean", "max"):
    print(f"  {_k:>6} statements/customer: {STATEMENT_COUNT_STATS[_k]}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: BUSINESS UNDERSTANDING -- MARKOV ROLL-RATE TRANSITION MODELING
# =============================================================================
_section("SECTION 6: Business Understanding -- Markov Roll-Rate Transition Modeling")

print(
    "PROBLEM 8 -- ROLL-RATE MODELING (bucket-transition probability model)\n\n"
    "Business case: Problems 6 and 7 both ask about a customer's CURRENT state -- 'is recent behavior "
    "different from full history' (Problem 6) or 'is the latest statement different from the customer's "
    "own baseline' (Problem 7). Problem 8 asks the classic credit-risk-operations question those two "
    "don't: 'given a customer is IN a given delinquency-severity state today, what is the real, empirically "
    "observed probability they move to each other state at their NEXT statement?' -- a first-order Markov "
    "transition-probability model, the standard 'roll-rate' technique collections and provisioning teams "
    "use to forecast how many accounts will roll from one severity bucket into a worse one next cycle.\n\n"
    "Concretely: every customer statement is assigned a discrete SEVERITY STATE (reusing Problem 4's own "
    "tier names -- Low Severity / Moderate Severity / Severe -- for cross-platform continuity). Sorting "
    "each customer's real statements by S_2 and pairing each statement with the one immediately following "
    "it yields real (state_t, state_t+1) observations; pooling these across the whole population gives a "
    "real, empirical transition-probability matrix -- no model is trained to predict the NEXT state "
    "directly; the matrix itself, estimated from real observed transitions, is the deliverable.\n\n"
    "METHODOLOGICAL NOTE (why this is NOT simply 'reuse Problem 4's tiers'): Problem 4's severity tiers "
    "are fit ONCE per customer from whole-history summary features (their _last value, _trend_slope, "
    "_trend_delta across ALL their statements) -- there is no way to ask 'what was customer X's tier at "
    "statement 3 of 7' from that artifact, because the summary features themselves are already collapsed "
    "across the customer's full history. A genuine transition model needs a state AT EACH STATEMENT, so "
    "this notebook defines a fresh, analogous per-STATEMENT severity score (same abs-correlation-weighted, "
    "direction-signed composite z-score METHODOLOGY Problem 4 established, reusing its real vetted feature "
    "universe) computed as a SNAPSHOT of each statement's own raw values -- fit fresh on the STATEMENT-"
    "level TRAIN population in Notebook 47, not reusing Problem 4's customer-level weights or cutpoints "
    "verbatim, since applying a fit calibrated to one population's distribution to a materially different "
    "population (customers vs. individual statements) would silently misclassify most rows.\n\n"
    "DATA-LIMITATION HONESTY (same standing caveat as Problems 5/6/7): this dataset provides exactly one "
    "eventual-default label per CUSTOMER, not a month-by-month ground truth of delinquency status -- there "
    "is no way to know whether a given historical statement was 'really' high-severity in a regulatory "
    "DPD-bucket sense. This notebook is honest about validating the fresh severity-state definition the "
    "only way the data supports: (a) does a customer's LATEST observed state -- the one point where the "
    "real target label is known -- show a real, monotonically increasing default rate from Low to Severe "
    "(Problem 4's own validation pattern, reused here); and (b) do customers whose state ESCALATED between "
    "their last two statements show a real, measurably different default rate than customers who did not "
    "escalate. Neither claims to know the 'true' DPD bucket of any historical statement -- only that the "
    "state definition and the transition matrix built from it carry real, measured predictive signal, "
    "reported plainly whichever way the real numbers point."
)
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: SEVERITY-STATE POLICY -- STATE NAMES AND CUT METHOD (ASSUMPTION)
# =============================================================================
_section("SECTION 7: Severity-State Policy -- State Names and Cut Method (ASSUMPTION)")

# ASSUMPTION: 3 states, reusing Problem 4's exact tier names for cross-
# platform continuity -- a customer or analyst already familiar with Problem
# 4's "Low/Moderate/Severe" vocabulary sees the same states here, just
# measured per-statement instead of per-customer-once. An alternative (e.g.
# 4-5 finer-grained DPD-style buckets) was considered and rejected for this
# pass: this dataset's raw D_* columns are anonymized delinquency indicators,
# not a literal days-past-due count, so finer buckets would imply a
# precision the source data doesn't honestly support.
STATE_NAMES = list(P4_TIER_ORDER)
N_STATES = len(STATE_NAMES)

# ASSUMPTION: tertile cuts, matching Problem 4's own convention exactly
# (33.333rd / 66.667th percentile). The CUT VALUES themselves are NOT set
# here -- they must be fit fresh on the real statement-level TRAIN population
# in Notebook 47 (see Section 6's methodological note above), so only the
# percentile CONVENTION is policy; the actual numbers are a measured output
# of Notebook 47, not an assumption of this notebook.
STATE_CUT_PERCENTILES = [33.333, 66.667]

# ASSUMPTION: a customer needs at least 2 real statements to contribute even
# ONE observed (state_t, state_t+1) transition pair -- this is the true
# minimum, not a padded threshold: a customer with exactly 2 statements
# still contributes one real, valid transition observation to the pooled
# population-level matrix (the matrix is estimated by pooling ALL customers'
# transitions together, not per customer, so no customer needs many
# transitions individually for the pooled estimate to be meaningful).
MIN_STATEMENTS_FOR_TRANSITION = 2

print(f"STATE_NAMES (ASSUMPTION, reused from Problem 4 for continuity): {STATE_NAMES}")
print(f"STATE_CUT_PERCENTILES (ASSUMPTION, matches Problem 4's tertile convention): {STATE_CUT_PERCENTILES}")
print(f"MIN_STATEMENTS_FOR_TRANSITION (ASSUMPTION, true minimum for 1 observed pair): {MIN_STATEMENTS_FOR_TRANSITION}")
print(f"Monitored feature universe (reused from Problem 4, real): {len(CANDIDATE_FEATURES)} base columns")
print("\n✅ Section 7 complete.")


# =============================================================================
# SECTION 8: KPI TARGETS -- HONEST, TECHNIQUE-APPROPRIATE (ASSUMPTION)
# =============================================================================
_section("SECTION 8: KPI Targets -- Honest, Technique-Appropriate (ASSUMPTION)")

# --- A Markov transition matrix is not a trained classifier -- there is no
#     single AUC-style number that tells you whether it's "good". This
#     notebook sets KPI targets appropriate to what a roll-rate model is
#     actually for: does the state definition carry real risk signal, and
#     does the fitted matrix behave coherently (real risk actually persists
#     and escalates, rather than looking like noise)? ---
RR_KPI_TARGETS = {
    "min_default_rate_ratio_top_to_bottom_tier": 1.5,
    "min_tier_population_pct": 15.0,
    "require_strict_monotonicity": True,
    "monotonicity_kpi_description": (
        "ASSUMPTION -- reused VERBATIM from Problem 4's own policy (docs/lgd_policy.json), for direct "
        "cross-platform comparability: PRIMARY, hard-gating KPI on whether the fresh per-statement "
        "severity-state definition carries real signal. Measured on each customer's LATEST observed "
        "state (the one point where the real target label is known): the real observed default rate must "
        "be strictly monotonically increasing Low < Moderate < Severe, the Severe-tier default rate must "
        "be >= 1.5x the Low-tier default rate, and each tier must hold >= 15% of the validation population "
        "(so no tier's default rate is a statistically meaningless number from a handful of customers)."
    ),
    "transition_matrix_coherence_check": {
        "description": (
            "NEW hard-gating KPI for this problem (no Problem 4/6/7 analogue -- a roll-rate model has no "
            "prior platform precedent to reuse): the real, empirically estimated transition matrix must "
            "show P(Severe -> Severe) [persistence in the worst state] STRICTLY GREATER than "
            "P(Low -> Severe) [a single-step jump from the best state to the worst]. This is the minimum "
            "sanity bar for 'this looks like a real risk process, not noise' -- if a customer's single-step "
            "jump probability from Low straight to Severe were higher than a Severe customer's probability "
            "of STAYING Severe, the fitted states would not be behaving as a coherent risk ordering at all."
        ),
        "hard_gate": True,
    },
    "escalation_validity_reporting": (
        "Not a pass/fail gate, an honest reporting requirement: Notebook 47 must report the real observed "
        "default rate among customers whose state ESCALATED between their last two statements, against the "
        "real observed default rate among customers whose state did NOT escalate over the same two "
        "statements -- reported plainly whichever direction the real numbers point, not gated on a target."
    ),
    "problem_6_stratification_requirement": (
        "Not a pass/fail gate, an honest reporting requirement: Notebook 47 must load Problem 6's real "
        "persisted trailing-window model and preprocessing artifacts, score each eligible customer's "
        "dynamic PD using their real trailing-W statements ENDING AT their last statement (the same point "
        "the final observed transition ends at, so the timing genuinely lines up -- reusing Problem 6's "
        "own build_trailing_window_store() feature logic verbatim, per this platform's established "
        "convention of copying rather than importing cross-notebook code), split the population at that "
        "score's real median, and report two separate final-transition (second-to-last -> last state) "
        "matrices for the above-median and below-median halves -- honestly noting throughout that "
        f"Problem 6's own model is currently "
        f"{'RECOMMENDED' if P6_RECOMMENDED_FOR_PRODUCTION else 'NOT RECOMMENDED'} FOR PRODUCTION, so this "
        "usage is explicitly exploratory (does the covariate add real information), not a claim that "
        "Problem 6's score is itself production-validated."
    ),
    "metrics_suite_requirement": (
        "STANDING RULE (user directive, 2026-08-25, carried from Problems 6/7): where a threshold-based "
        "binary prediction is meaningful (e.g. 'ESCALATED' as the predicted class in the escalation-"
        "validity check), Notebook 47/48 must compute and DISPLAY -- inline in the notebook AND in this "
        "problem's Word/Excel/HTML reports -- the full classification metrics suite: ROC-AUC, PR-AUC, "
        "Accuracy, Precision, Recall, F1, Specificity, Log Loss, Matthews Correlation Coefficient, and a "
        "full confusion matrix."
    ),
    "elevated_reporting_requirement": (
        "STANDING RULE (user directive, 2026-08-25, carried from Problem 7): Problem 8's Word report must "
        "synthesize MAXIMUM DETAIL from every one of this problem's notebooks (46-49), with a narrative "
        "'story' paragraph below every chart. Problem 8's HTML report must be an advanced, 'global "
        "standard' interactive dashboard with slicers, filters, full legends, and interactive KPI cards -- "
        "built in Notebook 49."
    ),
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "problem_4_reference": {"tier_order": P4_TIER_ORDER},
    "problem_6_reference": {"winning_w": P6_WINNING_W, "recommended_for_production": P6_RECOMMENDED_FOR_PRODUCTION},
}
for _k, _v in RR_KPI_TARGETS.items():
    if isinstance(_v, dict):
        print(f"{_k}: {json.dumps(_v)}")
    else:
        print(f"{_k}: {_v}")
print("\n✅ Section 8 complete.")


# =============================================================================
# SECTION 9: REAL TRANSITION-ELIGIBILITY COVERAGE
# =============================================================================
_section("SECTION 9: Real Transition-Eligibility Coverage")

# --- Real, measured coverage: what fraction of the platform's customers have
#     enough statements (>=MIN_STATEMENTS_FOR_TRANSITION) to contribute at
#     least one real transition pair. Computed from the SAME real per-
#     customer counts Section 5 already measured -- no re-scan needed. ---
_n_eligible = int((_counts_series >= MIN_STATEMENTS_FOR_TRANSITION).sum())
TRANSITION_ELIGIBILITY_COVERAGE_PCT = 100.0 * _n_eligible / _n_customers
_total_real_transition_pairs = int((_counts_series.clip(lower_bound=1) - 1).sum())
print(f"Customers with >= {MIN_STATEMENTS_FOR_TRANSITION} statements (real, measured): "
      f"{_n_eligible:,} / {_n_customers:,} ({TRANSITION_ELIGIBILITY_COVERAGE_PCT:.1f}%)")
print(f"Real total observable (state_t, state_t+1) transition pairs across the population "
      f"(sum of n_statements-1 per customer): {_total_real_transition_pairs:,}")
print(f"The remaining {_n_customers - _n_eligible:,} customers (a single statement only) are honestly "
      "EXCLUDED from the transition matrix in Notebook 47 -- not silently padded with a fabricated pair.")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: WRITE ROLL-RATE POLICY ARTIFACT
# =============================================================================
_section("SECTION 10: Write Roll-Rate Policy Artifact")

ROLL_RATE_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 8 -- Roll-Rate Modeling (Markov Transition-Probability Matrix Across Delinquency "
               "Severity States)",
    "state_names": STATE_NAMES,
    "n_states": N_STATES,
    "state_cut_percentiles": STATE_CUT_PERCENTILES,
    "min_statements_for_transition": MIN_STATEMENTS_FOR_TRANSITION,
    "transition_eligibility_coverage_pct": TRANSITION_ELIGIBILITY_COVERAGE_PCT,
    "total_real_transition_pairs_observable": _total_real_transition_pairs,
    "statement_count_stats": STATEMENT_COUNT_STATS,
    "monitored_features": {
        "features": CANDIDATE_FEATURES,
        "count": len(CANDIDATE_FEATURES),
        "source": "Reused from Problem 4's real severity-scoring bundle (severity_scoring_bundle.json, "
                   "base columns recovered from the suffix-tagged _last/_trend_delta/_trend_slope names) "
                   "-- not a fresh, unvetted selection. Scoring METHOD (abs-correlation-weighted composite "
                   "z-score) is reused from Problem 4; the fitted weights/means/stds/cutpoints are NOT "
                   "reused verbatim -- they are fit fresh on the statement-level population in Notebook 47.",
        "matches_problem_6_universe": _feature_universe_matches_p6,
    },
    "problem_6_covariate": {
        "model_path": str(P6_MODEL_PATH),
        "preprocessing_path": str(P6_PREPROCESSING_PATH),
        "winning_w": P6_WINNING_W,
        "recommended_for_production": P6_RECOMMENDED_FOR_PRODUCTION,
        "usage": "Exploratory stratification covariate only (Section 8's problem_6_stratification_requirement) "
                 "-- not a claim that Problem 6's own model is production-validated.",
    },
    "kpi_targets": RR_KPI_TARGETS,
    "ead_per_account_usd": EAD_PER_ACCOUNT_USD,
    "lgd_assumption": LGD_ASSUMPTION,
    "random_seed": RANDOM_SEED,
}
policy_path = RR_POLICY_DIR / "roll_rate_policy.json"
with open(policy_path, "w", encoding="utf-8") as f:
    json.dump(ROLL_RATE_POLICY, f, indent=2)
print(f"Wrote: {policy_path}")
print("\n✅ Section 10 complete.")


# =============================================================================
# SECTION 11: VERIFICATION -- INTEGRITY CHECKS
# =============================================================================
_section("SECTION 11: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Policy file was written", policy_path.exists())
_all_checks_passed &= _check("N_STATES matches len(STATE_NAMES)", N_STATES == len(STATE_NAMES))
_all_checks_passed &= _check("STATE_NAMES reused verbatim from Problem 4's real tier order",
                              STATE_NAMES == P4_TIER_ORDER)
_all_checks_passed &= _check("STATE_CUT_PERCENTILES is a strictly increasing pair in (0, 100)",
                              len(STATE_CUT_PERCENTILES) == 2
                              and 0 < STATE_CUT_PERCENTILES[0] < STATE_CUT_PERCENTILES[1] < 100)
_all_checks_passed &= _check("MIN_STATEMENTS_FOR_TRANSITION is the true minimum for 1 observed pair (>= 2)",
                              MIN_STATEMENTS_FOR_TRANSITION >= 2)
_all_checks_passed &= _check("Transition-eligibility coverage is a real measured percentage in (0, 100]",
                              0 < TRANSITION_ELIGIBILITY_COVERAGE_PCT <= 100)
_all_checks_passed &= _check("Total observable transition pairs is non-negative and consistent with "
                              "statement-count stats (cannot exceed n_customers * (max-1))",
                              0 <= _total_real_transition_pairs <= _n_customers * max(STATEMENT_COUNT_STATS["max"] - 1, 0))
_all_checks_passed &= _check("Statement count stats are internally consistent (min <= p25 <= max)",
                              STATEMENT_COUNT_STATS["min"] <= STATEMENT_COUNT_STATS["p25"] <= STATEMENT_COUNT_STATS["max"])
_all_checks_passed &= _check("Reused Problem 1's real champion AUC (not fabricated)",
                              FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"))
_all_checks_passed &= _check("Reused Problem 4's real monitored feature universe (no duplicates)",
                              len(CANDIDATE_FEATURES) == len(set(CANDIDATE_FEATURES)))
_all_checks_passed &= _check("Reused Problem 4's real monotonicity KPI values verbatim (1.5x / 15.0%)",
                              RR_KPI_TARGETS["min_default_rate_ratio_top_to_bottom_tier"] == 1.5
                              and RR_KPI_TARGETS["min_tier_population_pct"] == 15.0)
_all_checks_passed &= _check("EAD/LGD were inherited from Notebook 08, not re-guessed",
                              EAD_PER_ACCOUNT_USD == NB08_SUMMARY["ead_per_account_usd_assumption"]
                              and LGD_ASSUMPTION == NB08_SUMMARY["lgd_assumption"])
_all_checks_passed &= _check("Problem 6's real persisted model/preprocessing paths exist on disk",
                              P6_MODEL_PATH.exists() and P6_PREPROCESSING_PATH.exists())

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 11 complete -- all checks passed.")


# =============================================================================
# SECTION 12: WRITE NOTEBOOK 46 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 12: Write Notebook 46 Summary Artifact")

NB46_SUMMARY = {
    "notebook": "46_roll_rate_modeling_business_understanding.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "policy_path": str(policy_path),
    "state_names": STATE_NAMES,
    "n_states": N_STATES,
    "state_cut_percentiles": STATE_CUT_PERCENTILES,
    "min_statements_for_transition": MIN_STATEMENTS_FOR_TRANSITION,
    "transition_eligibility_coverage_pct": TRANSITION_ELIGIBILITY_COVERAGE_PCT,
    "total_real_transition_pairs_observable": _total_real_transition_pairs,
    "monitored_feature_count": len(CANDIDATE_FEATURES),
    "random_seed": RANDOM_SEED,
}
NB46_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_46_summary.json"
with open(NB46_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB46_SUMMARY, f, indent=2)
print(f"Wrote: {NB46_SUMMARY_PATH}")

_section("NOTEBOOK 46 COMPLETE")
print(f"STATE_NAMES (ASSUMPTION, reused from Problem 4)     : {STATE_NAMES}")
print(f"STATE_CUT_PERCENTILES (ASSUMPTION)                  : {STATE_CUT_PERCENTILES}")
print(f"MIN_STATEMENTS_FOR_TRANSITION (ASSUMPTION)          : {MIN_STATEMENTS_FOR_TRANSITION}")
print(f"Transition-eligibility coverage (real)              : {TRANSITION_ELIGIBILITY_COVERAGE_PCT:.1f}%")
print(f"Real observable transition pairs (population total) : {_total_real_transition_pairs:,}")
print(f"Monitored features (reused from Problem 4, real)    : {len(CANDIDATE_FEATURES)}")
print(f"Policy written to: {policy_path}")
print(
    "\nNext: 47_roll_rate_modeling_modeling.ipynb -- computes real per-statement severity scores/states, "
    "fits fresh tertile cutpoints on the statement-level TRAIN population, builds the real (state_t, "
    "state_t+1) transition matrix, validates the state definition against the real target label "
    "(monotonicity KPI + escalation-validity reporting), and reports the Problem-6-stratified final-"
    "transition comparison set in Section 8 above."
)
